CALLBACKS IN KERAS: AUTOMATIZZARE E MONITORARE IL TRAINING
(occhi e mani automatici del processo di training)

Le Callbacks in Keras servono per 'intercettare' quello che succede durante l'addestramento di una rete a reagire automaticamente.
Senza callback il training parte, fa 1000 epoche, termina, tu guardi DOPO cosa è successo.
Con le callback durante il training puoi, salvare il modello migliore, termare il training se sta peggiorando, ridurre il learning rate, scrivere log, fare monitoraggio, generare grafici, eseguire codice personalizzato.
Per esepio è possibile, durante il training, fermare il processo se la validation loss peggiora per 10 epoche, probabilmente stai overfittando.

1. EARLYSTOPPING
Serve per evitare overfitting e ridurre i tempi di training, è una delle più usate in assoluto

Esempio 1
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor='val_loss',
    mode='min'  #max nel caso di val_accuracy
    patience=5,
    restore_best_weights=True
)
model.fit(
    X_train,
    y_train,
    validation_data=(X_test, y_test),
    epochs=100,
    callbacks=[early_stop]
)

#aspetta 5 epoche di peggioramento validation loss, e torna automaticamente ai pesi migliori

2. MODELCHECKPOINT
Salva automaticamennte il modello migliore.
E' essenziale immagina 8 ore di training, carch pc, perdi tutto. Non perdi tutto se cha una callback di check point.
L'addestramento risiede nella RAM, se il computer si spegni perdi tutto, a meno che hai salvato sul disco rigido.
Parametri:
- Monitor: definisci la metrica da osservere, esempio val_loss
- Save Best Only: istruisce la callback a sovrascrivere il file solo se la metrica monitorata migliora rispetto alla precedente salvata.
- Mode: specifica se la metrica deve essere minimizzata (min) o massimizzata (max), adattandosi a loss o accuratezza

callbacks=[early_stop, checkpoint]

3. REDUCELRONPLATEAU
Se il modello smette di migliorare, abbassa automaticamente il learning rate.
All'inizio ho bisogno di un learning rate alto per apprendimento veloce, se manteniamo lo stesso lr rischi oscillazioni e di non convergere bene (plateau), allora ridue lr per un raffinamento più preciso.
Superea momento di stallo durante l'apprendimento

from tensorflow.keras.callbacks import ReduceLROnPlateau

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.1,
    patience=3
)
Se per 3 epoche il modello non miglioare, riduci il learning rate (lr) di 0.1% (lr=lr*0.4)

Parametri:
- Factor: il moltiplicatore applicato al lr attuale
- Patience: numero di epoche di atteza senza miglioramenti
- Min Delta: soglia minima di miglioramento, sotto la quale il progresso viene considerato nullo
- Min LR: limite inferiore del learning rate, per evitare paralisi modello

Ridurre il lr permette alla traiettoria di ottimizzare, di esplorare aree della loss più strette e profone.
Il parametro 'COOLDOWN' definisce il numero di epoche di tregue dopo una riduzione, prima di riprendere il normale monitoraggio della pazienza.
Attivando i log verbali, la callback notifica in console l'esatto istante in cui avviene il cambio di scala temporale dell'addestramento.

Un grafico di training mostra delle cadute improssive della loss, quasi a gradini, questi gradini indicano che la riduzione del lr è avvenuto.

4. TENSORBOARD
Serve per monitorare il training.
Puoi vedere loss, accuracy, grafi modello, pesi, embedding

from tensorflow.keras.callbacks import TensorBoard

tensorboard = TensorBoard(
    log_dir='logs'
)

Apri il browser e vedi dashboard training

tensorboard --logdir=logs

5. CSVLOGGER
Salva le metriche in CSV
Utile per analisi successive, confronto esperimenti

from tensorflow.keras.callbacks import CSVLogger

csv_logger = CSVLogger('training_log.csv')

COME SI USANO?
(le metto tutte insieme)

callbacks = [
    early_stop,
    checkpoint,
    reduce_lr,
    csv_logger
]

model.fit(
    X_train,
    y_train,
    validation_data=(X_test, y_test),
    epochs=100,
    callbacks=callbacks
)

6. CUSTOM CALLBACKS 
Crei tu una classe
Per esempio se ho bisogo di un sensore che Keras non ha previsto.
Le callback predefinite coprono i casi monuni, ma a volte serve eseguire azioni specifiche: inviare notifiche, loggare metriche su database esterni o modificare parametri al volo.
Keras permette di creare classi personalzzate ereditando da 'Callback' e sovrascrivendo metodi che vengono chiamati automaticamente dal motore di training.
Dove possono essere aggiunte queste custom:
- on_train_begin: logica eseguita una sola volta all'inizio del processo, utile per inizializzare file di log o timer
- on_epoch_end: il motodo più utilizzato, invocato dopo ogni epoca per analizzare le metriche e decidere azioni
- on_batch_begin: permette di intervenire su ogni singolo passaggio di dati, utile per monitoraggi al altissima frequenza
- Dizionario logs: il parametro passato ai metodi che contiene tutti i valori di loss e metriche correnti.

Con self.model dall'interno di una callback abbiamo accesso totale al modello: possiamo ispezionare i pesi, salvarli, o persino alterare il learning rate manualmente.
Self.params contiene informazioni, come il numero totale di epoche previsto o se è attivo il validation split.
Le custom callback sono il ponte perfetto per integrare librerie di terze parti come WandB o MLFlow all'interno del ciclo di fit di Keras

from tensorflow.keras.callbacks import Callback

class MiaCallback(Callback):

    def on_epoch_end(self, epoch, logs=None):
        print(f"Fine epoca {epoch}")
        print(logs)


model.fit(
    X_train,
    y_train,
    callbacks=[MiaCallback()]
)        

Quindi è possibile: inviare mail, salvare dati in DB, fare allert, fermare training, creare grafici, monitorare GPU, interagire con ERP/BI

Riassumendo:
EarlyStopping: evita overfitting
ModelCheckpoint: salva miglior modello
ReduceLROnPlateau: migliora convergenza
TensorBoard: monitora training
callback custom: automatizzano logica business

In [4]:
import tensorflow as tf
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.callbacks import Callback, ModelCheckpoint, ReduceLROnPlateau, EarlyStopping
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

# =================================================================
# 1. PREPARAZIONE DEL DATASET
# =================================================================
X, y = make_classification(n_samples=2000, n_features=30, n_informative=20, 
                           n_redundant=10, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# =================================================================
# 2. CUSTOM CALLBACK: LOG LR E MONITORAGGIO PESI
# =================================================================
class EnhancedLogger(Callback):
    def on_epoch_end(self, epoch, logs=None):
        # 1. Recupero Learning Rate
        lr = float(tf.keras.backend.get_value(self.model.optimizer.learning_rate))
        
        # 2. Monitoraggio Pesi (Prendiamo il primo strato Dense dopo l'Input)
        # Calcoliamo la media dei valori assoluti dei pesi per vedere quanto sono "grandi"
        weights, biases = self.model.layers[0].get_weights()
        avg_weight = np.mean(np.abs(weights))
        
        print(f"\n - [INFO] Fine Epoca {epoch+1}:")
        print(f"   > Learning Rate: {lr:.6f}")
        print(f"   > Media abs pesi (Layer 1): {avg_weight:.6f}")

# =================================================================
# 3. CONFIGURAZIONE CALLBACK
# =================================================================

# Salva il file del modello solo quando migliora la val_loss (min), andando a sovrascrivere il precedente
checkpoint_cb = ModelCheckpoint(
    filepath='miglior_modello.keras',
    monitor='val_loss',
    save_best_only=True,
    mode='min',
    verbose=1
)

# Riduce il LR se la loss stalla per 3 epoche, in modo che il modello si possa adattare meglio quando si avvicina al valore minimo
reduce_lr_cb = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.2,  # Riduce il LR del 20% quando la val_loss non migliora
    patience=3, #il modello non migliora per 3 epoche, riduci il LR
    min_lr=1e-6, #valore minimo del LR per evitare di scendere troppo
    verbose=1
)

# STOPPA il modello se non migliora per 10 epoche
early_stopping_cb = EarlyStopping(
    monitor='val_loss',
    patience=10,        # Numero di epoche da aspettare
    mode='min',
    restore_best_weights=True, # Al termine, ripristina i pesi migliori invece degli ultimi
    verbose=1
)

# =================================================================
# 4. COSTRUZIONE E TRAINING
# =================================================================
model = Sequential([
    Input(shape=(30,)),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dense(1, activation='sigmoid') #dens 1 perchè è un problema di classificazione binaria, sigmoid per avere output tra 0 e 1
    ])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

print("\nInizio addestramento con Early Stopping e Monitoraggio Pesi...")

history = model.fit(
    X_train, y_train,
    epochs=100, # Aumentiamo le epoche potenziali, tanto lo stop è automatico
    batch_size=32,
    validation_data=(X_val, y_val),
    callbacks=[checkpoint_cb, reduce_lr_cb, early_stopping_cb, EnhancedLogger()],
    verbose=0 
)


Inizio addestramento con Early Stopping e Monitoraggio Pesi...

Epoch 1: val_loss improved from None to 0.38211, saving model to miglior_modello.keras

Epoch 1: finished saving model to miglior_modello.keras

 - [INFO] Fine Epoca 1:
   > Learning Rate: 0.001000
   > Media abs pesi (Layer 1): 0.096811

Epoch 2: val_loss improved from 0.38211 to 0.30405, saving model to miglior_modello.keras

Epoch 2: finished saving model to miglior_modello.keras

 - [INFO] Fine Epoca 2:
   > Learning Rate: 0.001000
   > Media abs pesi (Layer 1): 0.097007

Epoch 3: val_loss improved from 0.30405 to 0.26373, saving model to miglior_modello.keras

Epoch 3: finished saving model to miglior_modello.keras

 - [INFO] Fine Epoca 3:
   > Learning Rate: 0.001000
   > Media abs pesi (Layer 1): 0.097220

Epoch 4: val_loss improved from 0.26373 to 0.22789, saving model to miglior_modello.keras

Epoch 4: finished saving model to miglior_modello.keras

 - [INFO] Fine Epoca 4:
   > Learning Rate: 0.001000
   > Media 

Aprendo il file abbiamo file .keras con il modello migliore, apribile con keras, per ricostruire il modello e riutilizzarlo